In [1]:
# install dependencies
!pip install chromadb sentence-transformers groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
import zipfile
import chromadb
import re
from sentence_transformers import SentenceTransformer
from groq import Groq

# Extract chroma_db
with zipfile.ZipFile('Files.zip', 'r') as zip_ref:
    zip_ref.extractall()


['a2b9da2c-275e-4438-ba23-b845979fb951', '54bd5337-5a78-41c0-b326-1fa8af803a84', 'chroma.sqlite3', '382332e0-4a4a-468e-9f2f-7a686c6c9c38']


In [ ]:
# Initialize embedding model and load ChromaDB
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

client = chromadb.PersistentClient(path="./chroma_db")
collection_global = client.get_or_create_collection(name="global_summaries")
collection_grouped = client.get_or_create_collection(name="grouped_summaries")
collection_raw = client.get_or_create_collection(name="raw_transactions")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total: 10135 chunks


In [ ]:
GROQ_API_KEY = "your_groq_api_key_here"
client_groq = Groq(api_key=GROQ_API_KEY)

In [ ]:
def extract_amount(text):
    match = re.search(r'\$[\d,]+\.?\d*', text)
    return float(match.group().replace('$', '').replace(',', '')) if match else 0

def extract_margin(text):
    match = re.search(r'Margin\s+([-\d.]+)%', text)
    return float(match.group(1)) if match else 0

def extract_discount(text):
    match = re.search(r'Average discount (\d+)%', text)
    return float(match.group(1)) if match else 0

def classify_query(query):
    q = query.lower()
    if "month" in q or "seasonal" in q or "november" in q or "december" in q:
        return "monthly", 15
    elif "year" in q and "trend" in q:
        return "yearly", 5
    elif "margin" in q and "year" in q:
        return "yearly", 5
    elif "category" in q and "revenue" in q:
        return "category", 5
    elif "sub-categ" in q or "sub categ" in q:
        return "subcategory", 8
    elif "region" in q:
        return "region", 5
    elif "state" in q or "states" in q:
        return "state", 8
    elif "city" in q or "cities" in q:
        return "city", 8
    elif "discount" in q or "frequently sold" in q:
        return "product_discount", 10
    else:
        return None, 5

def retrieve_context(query, num_results=5, fact_type=None):
    if fact_type == "subcategory" or fact_type == "monthly":
        retrieve_size = 100  
    elif "discount" in query.lower() or "frequently sold" in query.lower():
        retrieve_size = 2000
    else:
        retrieve_size = 25 if fact_type else num_results

    if "discount" in query.lower() or "frequently sold" in query.lower():
        grouped_results = collection_grouped.query(
            query_texts=["frequently discounted"],  
            n_results=2000,  
            include=["documents", "metadatas"])
        
        documents = []
        if grouped_results['documents'] and len(grouped_results['documents']) > 0:
            for doc in grouped_results['documents'][0]:
                if "Frequently discounted product" in doc:
                    documents.append(doc)
        
        documents = sorted(documents, key=extract_discount, reverse=True)
        documents = documents[:num_results]
        context = "\n".join(documents)
        return context[:2500] if len(context) > 2500 else context
    
    global_results = collection_global.query(
        query_texts=[query],
        n_results=retrieve_size,
        include=["documents", "metadatas"]
    )

    grouped_results = collection_grouped.query(
        query_texts=[query], 
        n_results=retrieve_size,
        include=["documents", "metadatas"])
    
    documents = []
    if global_results['documents'] and len(global_results['documents']) > 0:
        for doc, meta in zip(global_results['documents'][0], global_results['metadatas'][0]):
            if fact_type is not None and meta.get('fact_type') != fact_type:
                continue
            if not meta.get('is_aggregate', True):
                continue
            documents.append(doc)
    
    if grouped_results['documents'] and len(grouped_results['documents']) > 0:
        for doc, meta in zip(grouped_results['documents'][0], grouped_results['metadatas'][0]):
            if fact_type is not None and meta.get('fact_type') != fact_type:
                continue    
            documents.append(doc)
    
    if "margin" in query.lower():
        documents = sorted(documents, key=extract_margin, reverse=True)
    else:
        documents = sorted(documents, key=extract_amount, reverse=True)
    
    documents = documents[:num_results]
    context = "\n".join(documents)
    return context[:2500] if len(context) > 2500 else context

def rag_query(query):
    fact_type, num_results = classify_query(query)
    context = retrieve_context(query, num_results=num_results, fact_type=fact_type)

    if not context:
        return "No relevant data found."

    prompt = f"""You are a sales analytics expert analyzing a retail superstore dataset.

Your task: Answer the question PRECISELY and FACTUALLY based ONLY on the provided data.

DATA PROVIDED:
{context}

INSTRUCTIONS:
- Answer concisely (2-4 sentences maximum)
- Use exact numbers from the data
- Include dollar signs ($) and percentages (%)
- If the data doesn't contain the answer, say so clearly
- Do NOT make estimates or approximations
- Do NOT use hedging language like "appears to" or "seems to"
- Format multi-point answers as bullet points

QUESTION: {query}

ANSWER:"""

    response = client_groq.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.1-8b-instant",
        temperature=0.1,
        max_tokens=200
    )

    return response.choices[0].message.content


In [ ]:
test_queries = [
    "What is the sales trend over the 4-year period?",
    "Which months show the highest sales? Is there seasonality?",
    "How has profit margin changed over time?",
    "Which product category generates the most revenue?",
    "What sub-categories have the highest profit margins?",
    "Which products are frequently sold at a discount?",
    "Which region has the best sales performance?",
    "Compare sales performance across different states.",
    "Which cities are the top performers?",
    "Compare Technology vs Furniture sales trends.",
    "How does the West region compare to the East in terms of profit?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\nQ{i}: {query}")
    print("-" * 80)
    print(f"A: {rag_query(query)}\n")



[Query 1] (TREND)
Q: What is the sales trend over the 4-year period?
--------------------------------------------------------------------------------
A: Based on the provided data, the sales trend over the 4-year period is as follows:

• Sales increased from $470,532.51 in 2015 to $609,205.60 in 2016, a growth of $138,672.09 (29.5%).
• Sales then increased from $609,205.60 in 2016 to $733,215.26 in 2017, a growth of $123,989.66 (20.4%).
• The sales trend shows a consistent increase over the 4-year period, with a total growth of $262,682.75 (55.8%).


[Query 2] (TREND)
Q: Which months show the highest sales? Is there seasonality?
--------------------------------------------------------------------------------
A: Based on the provided data, the months with the highest sales are:

* September: $307,649.95
* October: $200,322.98

There is seasonality in sales, as the highest sales months are September and October, which are typically considered part of the holiday season or end-of-year sa

In [ ]:
def interactive_mode():
    print("Interactive RAG Mode - Type 'quit' to exit\n")
    while True:
        try:
            query = input(">>> Your question: ").strip()
            if query.lower() == 'quit':
                break
            if query:
                print(f"\n{rag_query(query)}\n")
        except KeyboardInterrupt:
            break

# Uncomment to run
# interactive_mode()
